# Step 3: standardise, cluster into regimes, add flags, and fit a first model

Scaling and clustering are fitted on the training period only and then applied to the test period. Cluster labels become one-hot regime flags alongside the raw features.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv(OUT_DIR / "model_dataset.csv", parse_dates=["entry_time"]).sort_values("entry_time")
feat = [c for c in df.columns if c[:3] in ("H1_", "H4_", "D1_")]
cut = df.entry_time.quantile(0.7)
train, test = df[df.entry_time <= cut], df[df.entry_time > cut]

In [ ]:
scaler = StandardScaler().fit(train[feat])
Xtr, Xte = scaler.transform(train[feat]), scaler.transform(test[feat])

regime_cols = [c for c in feat if c.startswith("D1_")]
idx = [feat.index(c) for c in regime_cols]
km = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Xtr[:, idx])


def with_flags(X, frame):
    labels = km.predict(X[:, idx])
    flags = pd.get_dummies(labels, prefix="regime").reindex(columns=[f"regime_{k}" for k in range(4)], fill_value=0)
    out = pd.DataFrame(X, columns=feat, index=frame.index)
    out[flags.columns] = flags.values
    out["direction"] = frame.direction.values
    for c in regime_cols:
        out[f"dir_x_{c}"] = out[c] * out.direction
    return out


Ftr, Fte = with_flags(Xtr, train), with_flags(Xte, test)
train.assign(regime=km.labels_).groupby("regime").win.agg(["count", "mean"])

In [ ]:
models = {
    "logistic": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED),
}
for name, m in models.items():
    m.fit(Ftr, train.win)
    print(f"{name:14s} test AUC = {roc_auc_score(test.win, m.predict_proba(Fte)[:, 1]):.3f}")
print(f"base rate (test) = {test.win.mean():.3f}")